# Streaming Dataset with IterableDataset — Solution

In [ ]:
import torch
from torch.utils.data import IterableDataset, DataLoader, get_worker_info
import os

torch.manual_seed(42)
N, D = 10_000, 5
X = torch.randn(N, D)
y = X @ torch.randn(D) + 0.1 * torch.randn(N)

data_path = "/tmp/streaming_test_data.csv"
with open(data_path, "w") as f:
    for i in range(N):
        row = ",".join(f"{v:.6f}" for v in X[i].tolist()) + f",{y[i]:.6f}\n"
        f.write(row)

In [ ]:
class StreamingDataset(IterableDataset):
    """Streams (x, y) pairs from a CSV file, one line at a time."""

    def __init__(self, filepath: str):
        super().__init__()
        self.filepath = filepath

    def __iter__(self):
        worker_info = get_worker_info()
        worker_id = 0 if worker_info is None else worker_info.id
        num_workers = 1 if worker_info is None else worker_info.num_workers

        with open(self.filepath) as f:
            for line_idx, line in enumerate(f):
                if line_idx % num_workers != worker_id:
                    continue
                values = [float(v) for v in line.strip().split(",")]
                x = torch.tensor(values[:-1])
                y = torch.tensor(values[-1])
                yield x, y


# Quick demo
ds = StreamingDataset(data_path)
loader = DataLoader(ds, batch_size=32)
bx, by = next(iter(loader))
print(f"Batch: X={bx.shape}, y={by.shape}")